# RETFound DR — Test Only từ file test.zip

Notebook này **chỉ đánh giá**, không train, không resume và không tạo pseudo-label. Quy trình: lấy `checkpoint-best.pth` và `test.zip` từ MyDrive, giải nén dữ liệu test, rồi lưu metrics/predictions/confusion matrix về Drive.

> Bật GPU trong Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch, sys, platform
from datetime import datetime, timezone
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Hãy chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')
print('=== RUNTIME DIAGNOSTICS ===')
print('Timestamp UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('cuDNN:', torch.backends.cudnn.version())

## 1. Gắn Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy code và cài thư viện

Cell này luôn chuyển vào đúng thư mục dự án trước khi gọi module `ai`, tránh lỗi `No module named ai`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/keras-grade-semi-supervised'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/grading/requirements-train.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Đã sẵn sàng tại {REPO_DIR} — branch {GITHUB_BRANCH}, commit {commit}')

## 3. Cấu hình duy nhất cần kiểm tra

Nếu checkpoint của bạn nằm chỗ khác, chỉ sửa `CHECKPOINT_PATH`. Kết quả sẽ được ghi vào `OUTPUT_DIR`.

In [ ]:
import json
# ===== CẤU HÌNH =====
CHECKPOINT_PATH = Path('/content/drive/MyDrive/checkpoint-best.pth')
TEST_ZIP_PATH = Path('/content/drive/MyDrive/test.zip')
OUTPUT_DIR = Path('/content/drive/MyDrive/test_results')
BATCH_SIZE = 2
NUM_WORKERS = 2
LIMIT_PER_CLASS = 0  # 0 = test toàn bộ; đặt 50 để chạy thử nhanh

if not CHECKPOINT_PATH.is_file():
    candidates = sorted(Path('/content/drive/MyDrive').glob('**/checkpoint-best.pth'))
    found = '\n'.join(f'  - {path}' for path in candidates) or '  (không tìm thấy file nào)'
    raise FileNotFoundError(
        f'Không tìm thấy checkpoint: {CHECKPOINT_PATH}\n'
        f'Các checkpoint-best.pth tìm thấy trên MyDrive:\n{found}\n'
        'Hãy sửa CHECKPOINT_PATH ở đầu cell này.'
    )
if not TEST_ZIP_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy dữ liệu test: {TEST_ZIP_PATH}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / 'notebook_test.log'
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
if not isinstance(checkpoint, dict):
    raise ValueError('Checkpoint không hợp lệ: cấp cao nhất phải là dictionary.')

checkpoint_method = checkpoint.get('method')
if checkpoint_method == 'fixed_support_target_domain_protonet':
    required_keys = {'encoder_model', 'prototypes', 'class_ids', 'base_model_args'}
    missing_keys = sorted(required_keys.difference(checkpoint))
    if missing_keys:
        raise ValueError(f'Checkpoint few-shot thiếu các khóa bắt buộc: {missing_keys}')
    CHECKPOINT_KIND = 'few_shot_protonet'
    EVALUATOR_MODULE = 'ai.grading.evaluate_fewshot'
elif {'model', 'args'}.issubset(checkpoint):
    CHECKPOINT_KIND = 'grading_classifier'
    EVALUATOR_MODULE = 'ai.grading.evaluate_test'
else:
    available_keys = sorted(map(str, checkpoint.keys()))
    raise ValueError(
        'Không nhận diện được định dạng checkpoint. Cần checkpoint grading có model+args '
        'hoặc checkpoint few-shot ProtoNet. '
        f'Các khóa hiện có: {available_keys}'
    )

checkpoint_summary = {
    'checkpoint': str(CHECKPOINT_PATH),
    'checkpoint_size_mb': round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2),
    'checkpoint_kind': CHECKPOINT_KIND,
    'checkpoint_method': checkpoint_method,
    'evaluator_module': EVALUATOR_MODULE,
    'epoch': checkpoint.get('epoch'),
    'best_qwk': checkpoint.get('best_qwk'),
    'best_support_loss': checkpoint.get('best_support_loss'),
    'args': checkpoint.get('args', checkpoint.get('base_model_args', {})),
    'grading_contract': checkpoint.get('grading_contract', {}),
    'test_zip': str(TEST_ZIP_PATH),
    'test_zip_size_mb': round(TEST_ZIP_PATH.stat().st_size / 1024**2, 2),
    'output_dir': str(OUTPUT_DIR),
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'limit_per_class': LIMIT_PER_CLASS,
}
print('=== TEST CONFIGURATION AND CHECKPOINT METADATA ===')
print(json.dumps(checkpoint_summary, indent=2, ensure_ascii=False, default=str))
LOG_PATH.write_text(json.dumps({'event': 'test_setup', 'timestamp_utc': datetime.now(timezone.utc).isoformat(), **checkpoint_summary}, ensure_ascii=False, default=str) + '\n', encoding='utf-8')
print('Detailed log:', LOG_PATH)

## 4. Giải nén và xác minh dữ liệu test

ZIP chấp nhận một trong ba cấu trúc: đầy đủ `train/validation/test`, chỉ có `test/0..4`, hoặc trực tiếp các thư mục lớp `0..4`. Chạy lại sẽ dùng bản đã giải nén nếu file ZIP không thay đổi.

In [ ]:
import shutil, zipfile

DATASET_EXTRACT_DIR = Path('/content/test_dataset')
marker = DATASET_EXTRACT_DIR / '.zip_identity'
zip_stat = TEST_ZIP_PATH.stat()
zip_identity = f'{zip_stat.st_size}:{zip_stat.st_mtime_ns}'
if marker.is_file() and marker.read_text(encoding='utf-8').strip() == zip_identity:
    print('Dùng lại dữ liệu đã giải nén:', DATASET_EXTRACT_DIR)
else:
    if DATASET_EXTRACT_DIR.exists():
        shutil.rmtree(DATASET_EXTRACT_DIR)
    DATASET_EXTRACT_DIR.mkdir(parents=True)
    print('Đang giải nén:', TEST_ZIP_PATH)
    with zipfile.ZipFile(TEST_ZIP_PATH) as archive:
        archive.extractall(DATASET_EXTRACT_DIR)
    marker.write_text(zip_identity, encoding='utf-8')

DATASET_DIR = DATASET_EXTRACT_DIR.resolve()
image_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp'}
image_paths = [path for path in DATASET_DIR.rglob('*') if path.is_file() and path.suffix.lower() in image_extensions]
n_images = len(image_paths)
if n_images == 0:
    raise ValueError(f'Không tìm thấy ảnh hỗ trợ trong {TEST_ZIP_PATH}')
class_counts = {grade: sum(1 for path in image_paths if path.parent.name == str(grade)) for grade in range(5)}
dataset_audit = {
    'event': 'dataset_audit',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_root': str(DATASET_DIR),
    'zip_identity': zip_identity,
    'total_images': n_images,
    'class_counts': class_counts,
    'extensions': sorted({path.suffix.lower() for path in image_paths}),
}
print('=== DATASET AUDIT ===')
print(json.dumps(dataset_audit, indent=2, ensure_ascii=False))
with LOG_PATH.open('a', encoding='utf-8') as log_file:
    log_file.write(json.dumps(dataset_audit, ensure_ascii=False) + '\n')
print('Cấu trúc lớp 0–4 sẽ được kiểm tra chặt chẽ khi bắt đầu test.')

## 5. Chạy test

Notebook tự chọn `ai.grading.evaluate_test` cho checkpoint grading hoặc `ai.grading.evaluate_fewshot` cho checkpoint ProtoNet; cả hai đều chỉ đánh giá, không train hoặc resume.

In [ ]:
def run_live(command):
    command.insert(1, '-u')
    started_at = datetime.now(timezone.utc)
    command_text = ' '.join(command)
    print('=== EVALUATION PROCESS START ===', flush=True)
    print('Started UTC:', started_at.isoformat(), flush=True)
    print('Working directory:', Path.cwd(), flush=True)
    print('Command:', command_text, flush=True)
    with LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(json.dumps({'event': 'evaluation_start', 'timestamp_utc': started_at.isoformat(), 'cwd': str(Path.cwd()), 'command': command_text}, ensure_ascii=False) + '\n')
    print('=' * 80, flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        with LOG_PATH.open('a', encoding='utf-8') as log_file:
            for line in process.stdout:
                timestamped = f'[{datetime.now(timezone.utc).isoformat()}] {line}'
                print(timestamped, end='', flush=True)
                log_file.write(timestamped)
                log_file.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finished_at = datetime.now(timezone.utc)
    completion = {'event': 'evaluation_end', 'timestamp_utc': finished_at.isoformat(), 'duration_seconds': round((finished_at - started_at).total_seconds(), 3), 'return_code': return_code}
    with LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(json.dumps(completion, ensure_ascii=False) + '\n')
    print('EVALUATION END:', json.dumps(completion, ensure_ascii=False), flush=True)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

cmd = [
    sys.executable, '-m', EVALUATOR_MODULE,
    '--checkpoint', str(CHECKPOINT_PATH),
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]
if LIMIT_PER_CLASS > 0:
    cmd.extend(['--limit-per-class', str(LIMIT_PER_CLASS)])
run_live(cmd)

## 6. Xem kết quả

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

metrics_path = OUTPUT_DIR / 'test_metrics.json'
predictions_path = OUTPUT_DIR / 'test_predictions.csv'
matrix_path = OUTPUT_DIR / 'confusion_matrix_normalized.png'
if not metrics_path.is_file():
    raise FileNotFoundError('Chưa có kết quả. Hãy chạy cell test ở trên trước.')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
predictions = pd.read_csv(predictions_path)
result_summary = {
    'event': 'evaluation_artifacts',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'metrics': metrics,
    'prediction_rows': len(predictions),
    'true_grade_distribution': predictions['true_grade'].value_counts().sort_index().to_dict(),
    'predicted_grade_distribution': predictions['predicted_grade'].value_counts().sort_index().to_dict(),
    'metrics_path': str(metrics_path),
    'predictions_path': str(predictions_path),
    'confusion_matrix_path': str(matrix_path),
}
print('=== FINAL EVALUATION SUMMARY ===')
print(json.dumps(result_summary, indent=2, ensure_ascii=False))
with LOG_PATH.open('a', encoding='utf-8') as log_file:
    log_file.write(json.dumps(result_summary, ensure_ascii=False, default=str) + '\n')
display(predictions.head(20))
display(Image(filename=str(matrix_path)))
print('Kết quả đã lưu lâu dài tại:', OUTPUT_DIR)